# Inflammatory Speech Flagger
**Capstone: AI-14 — Grazac Technologies / AI & ML NextGen Cohort**

Goal: train a classifier that flags potentially inflammatory Nigerian social media posts, returns a flag + a reason, and reports evaluation metrics with limitations.


## 1. Setup

In [ ]:
!pip install -q datasets scikit-learn pandas joblib python-dotenv

import pandas as pd
import numpy as np
import re
import joblib
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 2. Load the dataset

**AfriHate from Hugging Face.**
AfriHate ships the actual tweet text (not just IDs) for 15 African languages, annotated `Hate` / `Abusive` / `Normal` by native speakers. We use the Nigerian-relevant configs: Nigerian Pidgin (`pcm`), Hausa (`hau`), Igbo (`ibo`), and Yorùbá (`yor`). Source: https://huggingface.co/datasets/afrihate/afrihate

In [11]:
# load AfriHate (Nigerian-relevant languages) from Hugging Face
from datasets import load_dataset
from huggingface_hub import get_token, login
from dotenv import load_dotenv
import os

# Load the project .env file so HF_TOKEN is available without prompting.
load_dotenv(dotenv_path='.env')

nigerian_configs = ['pcm', 'hau', 'ibo', 'yor']  # Nigerian Pidgin, Hausa, Igbo, Yoruba
frames = []
for cfg in nigerian_configs:
    # First use the .env token, then any cached HF token, then a one-time prompt.
    token = os.getenv('HF_TOKEN') or get_token()
    if not token:
        token = input("Paste your Hugging Face read token (starts with 'hf_'): ").strip()
    if not token:
        raise ValueError("No Hugging Face token found. Add HF_TOKEN to .env or log in with `huggingface_hub.login()`.")

    login(token=token, add_to_git_credential=False)
    d = load_dataset('afrihate/afrihate', cfg, token=token)
    for split_name in d.keys():
        part = d[split_name].to_pandas()
        part['language'] = cfg
        frames.append(part)

df = pd.concat(frames, ignore_index=True)
print(df.columns.tolist())
print(df['language'].value_counts())
df.head()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


['id', 'tweet', 'label', 'language', 'length']
language
pcm    10599
hau     6644
ibo     5003
yor     4879
Name: count, dtype: int64


,id,tweet,label,language,length
0,train_nigerian_pidgin_00001,open on the island na you really dont expect m...,Normal,pcm,NaN
1,train_nigerian_pidgin_00002,"@USER @USER I'm in Akwa ibom state, you guys r...",Abuse,pcm,NaN
2,train_nigerian_pidgin_00003,pastor dey shout your children will suck your ...,Normal,pcm,NaN
3,train_nigerian_pidgin_00004,this hausa keke marwa rider just dey form need...,Normal,pcm,NaN
4,train_nigerian_pidgin_00005,"@USER @USER They citizens, wannabe call patrio...",Hate,pcm,NaN


In [12]:
# AfriHate's label column has string values: 'Hate', 'Abuse', 'Normal' (case may vary).
# We treat hate + abuse as 'inflammatory' (1) and normal as 'not' (0).
TEXT_COL = 'tweet' if 'tweet' in df.columns else 'text'

# Sanity check FIRST: see the exact raw label strings before mapping, so nothing
# silently falls through into the wrong bucket.
print('Raw label values in this dataset:', df['label'].unique())

df = df[[TEXT_COL, 'label']].rename(columns={TEXT_COL: 'text', 'label': 'raw_label'})
df = df.dropna(subset=['text', 'raw_label'])
df['raw_label'] = df['raw_label'].astype(str).str.lower().str.strip()
df['label'] = df['raw_label'].apply(lambda x: 1 if x in ('hate', 'abuse', 'abusive') else 0)

# Second sanity check: confirm no raw label slipped through unmapped/unexpected.
unmapped = df.loc[~df['raw_label'].isin(['hate', 'abuse', 'abusive', 'normal']), 'raw_label'].unique()
if len(unmapped) > 0:
    print('WARNING: unexpected label values found, check these:', unmapped)

df = df[['text', 'label']].drop_duplicates().reset_index(drop=True)
print(df['label'].value_counts())
print(f'Total posts: {len(df)}')
df.head()

Raw label values in this dataset: ['Normal' 'Abuse' 'Hate']
label
1    14966
0    10907
Name: count, dtype: int64
Total posts: 25873


,text,label
0,open on the island na you really dont expect m...,0
1,"@USER @USER I'm in Akwa ibom state, you guys r...",1
2,pastor dey shout your children will suck your ...,0
3,this hausa keke marwa rider just dey form need...,0
4,"@USER @USER They citizens, wannabe call patrio...",1


## 3. Preprocess text

We clean lightly on purpose: Nigerian Pidgin and code-switched slang carry meaning, so we avoid aggressive normalization that would strip that out.

In [13]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)      # remove URLs
    text = re.sub(r'@\w+', ' ', text)                    # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)                 # keep hashtag words, drop the '#'
    text = re.sub(r'[^a-z0-9\s]', ' ', text)              # strip punctuation/emoji
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
df[['text', 'clean_text', 'label']].head()

,text,clean_text,label
0,open on the island na you really dont expect m...,open on the island na you really dont expect m...,0
1,"@USER @USER I'm in Akwa ibom state, you guys r...",i m in akwa ibom state you guys r very stupid ...,1
2,pastor dey shout your children will suck your ...,pastor dey shout your children will suck your ...,0
3,this hausa keke marwa rider just dey form need...,this hausa keke marwa rider just dey form need...,0
4,"@USER @USER They citizens, wannabe call patrio...",they citizens wannabe call patriotic ones,1


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

Train: 20698  Test: 5175


## 4. Feature extraction + model training

In [15]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_vec, y_train)
print('Model trained.')

Model trained.


## 5. Evaluation

In [16]:
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred, target_names=['not_inflammatory', 'inflammatory']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.3f}")

                  precision    recall  f1-score   support

not_inflammatory       0.73      0.83      0.78      2182
    inflammatory       0.86      0.77      0.82      2993

        accuracy                           0.80      5175
       macro avg       0.80      0.80      0.80      5175
    weighted avg       0.81      0.80      0.80      5175

Confusion matrix:
[[1817  365]
 [ 674 2319]]
Macro F1: 0.797


## 6. Flag + reason output

The spec asks for a flag AND a reason, not just a label. We combine two signals:
1. **Model confidence** from Logistic Regression's predicted probability.
2. **Top contributing words** — the words in the post with the highest learned weight toward the 'inflammatory' class, which act as a human-readable reason.

This is a simple, explainable approach appropriate for a baseline capstone model.

In [17]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_[0]

def explain_prediction(raw_text: str, top_k: int = 3):
    """Return flag (bool), probability, and the top words driving the decision."""
    cleaned = clean_text(raw_text)
    vec = vectorizer.transform([cleaned])
    prob = model.predict_proba(vec)[0][1]
    flag = bool(prob >= 0.5)

    present_idx = vec.nonzero()[1]
    if len(present_idx) == 0:
        return flag, prob, []

    word_scores = [(feature_names[i], coefs[i]) for i in present_idx]
    word_scores.sort(key=lambda x: x[1], reverse=flag)  # push toward the predicted class
    top_words = [w for w, s in word_scores[:top_k]]
    return flag, prob, top_words

def flag_post(raw_text: str) -> dict:
    flag, prob, top_words = explain_prediction(raw_text)
    if flag:
        reason = f"Flagged as inflammatory (confidence {prob:.2f}). Contributing terms: {', '.join(top_words) if top_words else 'n/a'}."
    else:
        reason = f"Not flagged (confidence {1 - prob:.2f} not inflammatory)."
    return {'text': raw_text, 'flag': flag, 'probability': round(float(prob), 3), 'reason': reason}

# Demo
samples = [
    "This particular tribe don spoil this country, una all no good",
    "Congrats to everyone graduating today, God bless una plenty",
]
for s in samples:
    print(flag_post(s))

{'text': 'This particular tribe don spoil this country, una all no good', 'flag': True, 'probability': 0.783, 'reason': 'Flagged as inflammatory (confidence 0.78). Contributing terms: country, spoil, all.'}
{'text': 'Congrats to everyone graduating today, God bless una plenty', 'flag': False, 'probability': 0.213, 'reason': 'Not flagged (confidence 0.79 not inflammatory).'}


## 7. Save the trained model

In [18]:
import os
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/inflammatory_flagger_model.joblib')
joblib.dump(vectorizer, 'models/tfidf_vectorizer.joblib')
print('Saved model + vectorizer to /models')

# In Colab, download them to your machine, or save to Drive:
# from google.colab import files
# files.download('models/inflammatory_flagger_model.joblib')
# files.download('models/tfidf_vectorizer.joblib')

Saved model + vectorizer to /models


## 8. Limitations 
- **Dataset scope**: NaijaHate is Twitter/X-only; posts from other platforms (Facebook, WhatsApp, TikTok) may look different and the model may not generalize well.
- **Class imbalance**: Neutral posts vastly outnumber hateful ones in real-world samples, which can bias precision/recall — we used `class_weight='balanced'` to partially correct this, but it's not a full fix.
- **Code-switching & dialects**: Hausa, Yoruba, Igbo, and heavy Pidgin-English code-mixing are only partially represented; TF-IDF + Logistic Regression has no real understanding of context or sarcasm.
- **Explainability is shallow**: 'Top contributing words' is a proxy for a reason, not true reasoning — a word can be flagged out of context (e.g. reclaimed slang, quotes, satire).
- **Bias risk**: any hate-speech classifier trained on human-annotated data inherits annotator bias; false positives can silence legitimate speech, false negatives can miss real harm.
